In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from os.path import exists

sys.path.append('../..')

In [3]:
import numpy as np
from loguru import logger

from stable_baselines3 import PPO, DQN

In [4]:
from vimms.Common import POSITIVE, set_log_level_warning, load_obj, save_obj
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler, MZMLChromatogramSampler
from vimms.Noise import UniformSpikeNoise
from vimms.Evaluation import evaluate_real
from vimms.Chemicals import ChemicalMixtureFromMZML
from vimms.Roi import RoiBuilderParams

from mass_spec_utils.data_import.mzmine import load_picked_boxes

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.evaluation import evaluate, run_method

/opt/anaconda3/envs/vimms-gym/lib/python3.9/site-packages/statsmodels/compat/pandas.py:61: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import Int64Index as NumericIndex
/opt/anaconda3/envs/vimms-gym/lib/python3.9/site-packages/psims/mzmlb/writer.py:15: UserWarning: hdf5plugin is missing! Only the slower GZIP compression scheme will be available! Please install hdf5plugin to be able to use Blosc.
  warnings.warn(


# 1. Parameters

In [5]:
n_chemicals = (5000, 20000)
mz_range = (70, 1000)
rt_range = (0, 1440)
intensity_range = (1E4, 1E20)

In [6]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [7]:
isolation_window = 0.7
N = 10
rt_tol = 120
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

enable_spike_noise = True
noise_density = 0.1
noise_max_val = 1e3

In [8]:

mzml_filename = '../fullscan_QCB.mzML'
samplers = None
samplers_pickle = 'samplers_fullscan_QCB.mzML.p'
if exists(samplers_pickle):
    logger.info('Loaded %s' % samplers_pickle)
    samplers = load_obj(samplers_pickle)
    mz_sampler = samplers['mz']
    ri_sampler = samplers['rt_intensity']
    cr_sampler = samplers['chromatogram']
else:
    logger.info('Creating samplers from %s' % mzml_filename)
    mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
    ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                           min_log_intensity=min_log_intensity,
                                           max_log_intensity=max_log_intensity)
    roi_params = RoiBuilderParams(min_roi_length=3, at_least_one_point_above=1000)
    cr_sampler = MZMLChromatogramSampler(mzml_filename, roi_params=roi_params)
    samplers = {
        'mz': mz_sampler,
        'rt_intensity': ri_sampler,
        'chromatogram': cr_sampler
    }
    save_obj(samplers, samplers_pickle)

2022-04-26 11:32:21.933 | INFO     | __main__:<module>:5 - Loaded samplers_fullscan_QCB.mzML.p


In [9]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
        'cr_sampler': cr_sampler,
    },
    'noise': {
        'enable_spike_noise': enable_spike_noise,
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [10]:
max_peaks = 200
in_dir = 'results'

In [11]:
n_eval_episodes = 10
deterministic = True

# 2. Evaluation

#### Generate some chemical sets

In [12]:
n_eval_episodes = 1
eval_dir = 'evaluation'
methods = [
    'PPO',
    'TopN',
    'random',    
]

In [13]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(n_eval_episodes):
    print(i)
    chems = generate_chemicals(chemical_creator_params)
    chem_list.append(chems)

0


2022-04-26 11:32:27.981 | DEBUG    | vimms.Chemicals:sample:468 - Sampled rt and intensity values and chromatograms


#### Run different methods

In [14]:
len(chem_list[0])

11767

In [15]:
max_peaks

200

In [16]:
out_dir = eval_dir
in_dir, out_dir

('results', 'evaluation')

#### Compare to Top-10

In [17]:
idx = 10

In [18]:
copy_params = dict(params)
copy_params['env']['rt_tol'] = rt_tol

for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()
    
    if method == 'PPO':
        fname = os.path.join(in_dir, 'model_%s_%d.zip' % (method, idx))
        model = PPO.load(fname)
    elif method == 'DQN':
        fname = os.path.join(in_dir, 'model_%s%d.zip' % (method, idx))
        model = DQN.load(fname)
    else:
        model = None

    env = DDAEnv(max_peaks, copy_params)
    run_method(env, chem_list, method, out_dir, min_ms1_intensity=min_ms1_intensity, model=model, N=10, print_eval=True)
    print()

method = PPO max_peaks = 200 rt_tol = 120

Episode 0 finished after 5445 timesteps with reward 775.8618478042712
{'coverage_prop': '0.315', 'intensity_prop': '0.220', 'ms1/ms2 ratio': '0.476', 'efficiency': '1.004'}

method = TopN max_peaks = 200 rt_tol = 120

Episode 0 finished after 5876 timesteps with reward 828.7132085044194
{'coverage_prop': '0.382', 'intensity_prop': '0.265', 'ms1/ms2 ratio': '0.291', 'efficiency': '0.987'}

method = random max_peaks = 200 rt_tol = 120

Episode 0 finished after 7155 timesteps with reward 191.9469188810652
{'coverage_prop': '0.158', 'intensity_prop': '0.107', 'ms1/ms2 ratio': '0.006', 'efficiency': '0.261'}



#### Test classic Top-N controller in ViMMS

In [19]:
from vimms.MassSpec import IndependentMassSpectrometer
from vimms.Controller import TopNController, TopN_SmartRoiController, WeightedDEWController
from vimms.Environment import Environment

In [20]:
chemicals = chem_list[0]
len(chemicals)

11767

In [21]:
set_log_level_warning()

7

In [22]:
noise_params = params['noise']
noise_density = noise_params['noise_density']
noise_max_val = noise_params['noise_max_val']
noise_min_mz = noise_params['mz_range'][0]
noise_max_mz = noise_params['mz_range'][1]
spike_noise = UniformSpikeNoise(noise_density, noise_max_val, min_mz=noise_min_mz,
                                max_mz=noise_max_mz)
mass_spec = IndependentMassSpectrometer(ionisation_mode, chems, spike_noise=spike_noise)

In [23]:
controller = TopNController(ionisation_mode, N, isolation_window, mz_tol, rt_tol, min_ms1_intensity)
env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=True, out_dir=out_dir, out_file='TopN_controller.mzML')
env.run()

  0%|          | 0/1440 [00:00<?, ?it/s]

In [24]:
evaluate(env)

{'coverage_prop': '0.381',
 'intensity_prop': '0.266',
 'ms1/ms2 ratio': '0.292',
 'efficiency': '0.987'}

### Test on QCB data

#### Load pre-processed QCB chemicals

In [25]:
fullscan_file = '../fullscan_QCB.mzML'

In [26]:
# min_roi_intensity = 0
# min_roi_length = 0

min_roi_intensity = 500
min_roi_length = 0
at_least_one_point_above = 5000

In [27]:
filename = 'datasets_%d_%d_%d.p' % (min_roi_intensity, min_roi_length, at_least_one_point_above)

if exists(filename):
    chemicals = load_obj(filename)
    print(len(chemicals))
else:
    rp = RoiBuilderParams(min_roi_intensity=min_roi_intensity, min_roi_length=min_roi_length, 
                   at_least_one_point_above=at_least_one_point_above)
    cm = ChemicalMixtureFromMZML(fullscan_file, roi_params=rp)
    chemicals = cm.sample(None, 2, source_polarity=ionisation_mode)
    print(len(chemicals))
    save_obj(chemicals, filename)

19889


#### Filter chemicals by mz and RT range

In [28]:
filtered = []
for chem in chemicals:
    if (min_mz < chem.isotopes[0][0] < max_mz) and (min_rt < chem.rt < max_rt):
        filtered.append(chem)
        
len(filtered)

19823

In [29]:
filtered_chem_list = [filtered]

#### Disable spike noise in DDAEnv

In [30]:
copy_params = dict(params)
copy_params['noise']['enable_spike_noise'] = False
copy_params

{'chemical_creator': {'mz_range': (70, 1000),
  'rt_range': (0, 1440),
  'intensity_range': (10000.0, 1e+20),
  'n_chemicals': (5000, 20000),
  'mz_sampler': <vimms.ChemicalSamplers.MZMLFormulaSampler at 0x7fc0da2b2d00>,
  'ri_sampler': <vimms.ChemicalSamplers.MZMLRTandIntensitySampler at 0x7fc0fa2f1880>,
  'cr_sampler': <vimms.ChemicalSamplers.MZMLChromatogramSampler at 0x7fc0eabf7ee0>},
 'noise': {'enable_spike_noise': False,
  'noise_density': 0.1,
  'noise_max_val': 1000.0,
  'mz_range': (70, 1000)},
 'env': {'ionisation_mode': 'Positive',
  'rt_range': (0, 1440),
  'isolation_window': 0.7,
  'mz_tol': 10,
  'rt_tol': 120}}

#### Test different methods

In [31]:
for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()
    
    if method == 'PPO':
        fname = os.path.join(in_dir, 'model_%s_%d.zip' % (method, idx))
        model = PPO.load(fname)
    elif method == 'DQN':
        fname = os.path.join(in_dir, 'model_%s%d.zip' % (method, idx))
        model = DQN.load(fname)
    else:
        model = None

    env = DDAEnv(max_peaks, copy_params)
    run_method(env, filtered_chem_list, method, out_dir, min_ms1_intensity=min_ms1_intensity, model=model, 
               N=10, mzml_prefix='QCB')
    print()

method = PPO max_peaks = 200 rt_tol = 120

Episode 0 finished after 5074 timesteps with reward 339.59885236853904

method = TopN max_peaks = 200 rt_tol = 120

Episode 0 finished after 5022 timesteps with reward 436.6262891694092

method = random max_peaks = 200 rt_tol = 120

Episode 0 finished after 7141 timesteps with reward 359.9071662743003



#### Test classic Top-N controller in ViMMS

In [32]:
mass_spec = IndependentMassSpectrometer(ionisation_mode, filtered)

In [33]:
controller = TopNController(ionisation_mode, N, isolation_window, mz_tol, rt_tol, min_ms1_intensity)
env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=True, out_dir=out_dir, 
                  out_file='QCB_TopN_controller.mzML')
env.run()

  0%|          | 0/1440 [00:00<?, ?it/s]

#### Evaluation from mzML

Evaluation against the list of peaks picked from the fullscan QCB files.

There are two sets of parameters used for the peak picking: 'before' and 'after'.
'After' is more strict than 'before', with higher thresholds on intensity etc.

In [35]:
fullscan_path = os.path.abspath('../fullscan_QCB.mzML')
fullscan_path

'/Users/joewandy/Work/git/vimms-gym/notebooks/fullscan_QCB.mzML'

### 'Before' results

In [34]:
aligned_file = os.path.abspath('../fullscan_QCB_box_before.csv')
aligned_file

'/Users/joewandy/Work/git/vimms-gym/notebooks/fullscan_QCB_box_before.csv'

In [36]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_PPO_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.23950149011108102], [0.15736423887055373])

In [37]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.41641831481983205], [0.27549609875746334])

In [38]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_controller.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.4177729612571119], [0.2774318956639041])

In [39]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_random_0.mzML'),  
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.13709021945272284], [0.3038820162740202])

### 'After' results

In [40]:
aligned_file = os.path.abspath('../fullscan_QCB_box_after.csv')
aligned_file

'/Users/joewandy/Work/git/vimms-gym/notebooks/fullscan_QCB_box_after.csv'

In [41]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_PPO_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.6090225563909775], [0.3254429945499315])

In [42]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.8020050125313283], [0.35524311748378756])

In [43]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_controller.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.7969924812030075], [0.3501550493608378])

In [44]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_random_0.mzML'),  
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.5413533834586466], [0.5178471210848645])